# Task 4: Neo4j Kafka Sink Connector & Replay-Safe Graph Ingestion

Tài liệu này ghi nhận quá trình cấu hình, kiểm thử, và vận hành Task 4 trong dự án Nhập môn Dữ liệu lớn.

## 1. Mục tiêu & Các Ràng Buộc Thiết Kế
- **Replay-Safe (Idempotency)**: Đảm bảo khi phát lại (replay) các sự kiện, đồ thị không bị nhân đôi (duplicate) và không bị sai lệch cấu trúc.
- **Stale Event Policy**: Các sự kiện lỗi thời (stale events) do mạng chậm hoặc replay không được ghi đè lên trạng thái mới của đồ thị.
- **Edge-Before-Node Handling**: Khi một quan hệ (Edge) được ghi nhận trước nút nguồn/đích của nó, connector tự động tạo các nút placeholder để giữ tính toàn vẹn tham chiếu đồ thị.
- **Dead Letter Queue (DLQ)**: Các record lỗi cú pháp hoặc hỏng schema được chuyển hướng đến topic `connector.errors` mà không làm ngắt quãng pipeline.
- **Compose Isolation**: Neo4j và Kafka Connect chạy độc lập trong `infra/docker-compose.neo4j.yml` để không ảnh hưởng đến base Compose.

## 2. Thiết Lập Môi Trường

In [1]:
import os
import subprocess
import json
from pathlib import Path

PROJECT_ROOT = Path(
    subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
)
print("PROJECT_ROOT:", PROJECT_ROOT)

os.chdir(PROJECT_ROOT)

PROJECT_ROOT: /home/phat/AI_Project/lab04-cpg-streaming


## 3. Kiểm Tra Trạng Thái Connector Đang Hoạt Động

In [2]:
# Xem danh sách connector đang chạy
res = subprocess.run(["curl", "-s", "http://localhost:8083/connectors"], capture_output=True, text=True, check=True)
print("Active Connectors:", json.loads(res.stdout))

Active Connectors: ['neo4j-nodes-sink', 'neo4j-edges-sink']


## 4. Kiểm Tra Chi Tiết Cấu Hình Nodes Sink Connector

In [3]:
res = subprocess.run(["curl", "-s", "http://localhost:8083/connectors/neo4j-nodes-sink/config"], capture_output=True, text=True, check=True)
config = json.loads(res.stdout)
print(json.dumps(config, indent=2))

{
  "connector.class": "streams.kafka.connect.sink.Neo4jSinkConnector",
  "neo4j.connection.liveness.check.timeout.msecs": "5000",
  "neo4j.authentication.basic.password": "Lab04Neo4jLocal123",
  "topics": "cpg.nodes",
  "neo4j.topic.cypher.cpg.nodes": "WITH event OPTIONAL MATCH (n_old:CPGNode {id: event.node.node_id}) FOREACH (x IN CASE WHEN event.event_type = 'NODE_UPSERT' THEN [1] ELSE [] END | MERGE (n:CPGNode {id: event.node.node_id}) ON CREATE SET n += coalesce(event.node.properties, {}), n.id = event.node.node_id, n.placeholder = false, n.node_type = event.node.node_type, n.name = event.node.name, n.qualified_name = event.node.qualified_name, n.ast_path = event.node.ast_path, n.line_start = event.node.line_start, n.column_start = event.node.column_start, n.line_end = event.node.line_end, n.column_end = event.node.column_end, n.repository_id = event.repository_id, n.commit_sha = event.commit_sha, n.file_path = event.file_path, n.file_id = event.file_id, n.parser_version = event.p

## 5. Kiểm Tra Chi Tiết Cấu Hình Edges Sink Connector

In [4]:
res = subprocess.run(["curl", "-s", "http://localhost:8083/connectors/neo4j-edges-sink/config"], capture_output=True, text=True, check=True)
config = json.loads(res.stdout)
print(json.dumps(config, indent=2))

{
  "connector.class": "streams.kafka.connect.sink.Neo4jSinkConnector",
  "neo4j.connection.liveness.check.timeout.msecs": "5000",
  "neo4j.topic.cypher.cpg.edges": "WITH event OPTIONAL MATCH (s_exist)-[r_exist:CPG_EDGE {edge_id: event.edge.edge_id}]->(d_exist) WITH event, s_exist, d_exist, r_exist, (CASE WHEN r_exist IS NOT NULL AND (s_exist.id <> event.edge.source_id OR d_exist.id <> event.edge.target_id) THEN true ELSE false END) AS mismatch WITH event, mismatch, (1 / (CASE WHEN mismatch THEN 0 ELSE 1 END)) AS ignore_me, r_exist WHERE NOT mismatch OPTIONAL MATCH (t_src:CPGNodeTombstone {id: event.edge.source_id, generation_id: event.file_id + ':' + event.content_hash + ':' + event.parser_version + ':' + event.schema_version}) OPTIONAL MATCH (t_dst:CPGNodeTombstone {id: event.edge.target_id, generation_id: event.file_id + ':' + event.content_hash + ':' + event.parser_version + ':' + event.schema_version}) OPTIONAL MATCH (te:CPGEdgeTombstone {id: event.edge.edge_id, generation_id: eve

## 6. Chạy Script Inspect Kiểm Tra Số Lượng Nodes & Edges Thực Tế Trong Đồ Thị

In [5]:
res = subprocess.run(["python", "scripts/inspect_neo4j_graph.py"], capture_output=True, text=True, check=True)
print(json.dumps(json.loads(res.stdout), indent=2))

{
  "node_count_by_file": [
    {
      "file_id": "45d86c35c8e49a3a96e497f546058468eaf925d48e98c036b41c1d7babc36b66",
      "node_count": 7
    },
    {
      "file_id": "test_mixedbatch_2c8a922a",
      "node_count": 6
    },
    {
      "file_id": "test_mixedbatch_9ccd90b5",
      "node_count": 6
    },
    {
      "file_id": "test_mixedbatch_8f630f57",
      "node_count": 6
    },
    {
      "file_id": "test_mixedbatch_83764739",
      "node_count": 6
    },
    {
      "file_id": "consistency_test_file_id",
      "node_count": 5
    },
    {
      "file_id": "test_edge_mismatch_file_id",
      "node_count": 2
    },
    {
      "file_id": "test_routing_file_id",
      "node_count": 2
    },
    {
      "file_id": "test_edge_newgen_file_id",
      "node_count": 2
    },
    {
      "file_id": "test_edge_del_replay_file_id",
      "node_count": 2
    }
  ],
  "relationship_count_by_file": [
    {
      "file_id": "45d86c35c8e49a3a96e497f546058468eaf925d48e98c036b41c1d7babc36b66",
 

## 7. Giải Thích Thuật Toán Trích Xuất & Xử Lý Replay-Safe

- **Toán tử Nhánh (`FOREACH`)**: Để tránh giới hạn của `CALL { ... }` subquery (khi không tìm thấy bản ghi có thể làm dừng toàn bộ luồng xử lý), chúng tôi sử dụng toán tử `FOREACH (x IN CASE WHEN event.event_type = '...' THEN [1] ELSE [] END | ...)` làm cấu trúc rẽ nhánh điều kiện cho việc thực thi các lệnh Cypher tương ứng với `UPSERT` hoặc `DELETE`.
- **Node và Edge Tombstones**: 
  * Khi thực hiện xóa Node (`NODE_DELETE`), hệ thống tự động tạo `CPGNodeTombstone` ghi nhận trạng thái đã xóa của Node đó.
  * Khi thực hiện xóa Edge (`EDGE_DELETE`), hệ thống luôn tạo `CPGEdgeTombstone` từ các trường thông tin của sự kiện (ngay cả khi relationship chưa tồn tại trong Neo4j).
  * Trong các truy vấn `EDGE_UPSERT` hoặc `NODE_UPSERT`, hệ thống kiểm tra sự tồn tại của các Tombstone tương ứng với cùng định danh thế hệ (`generation_id`). Nếu tombstone cùng thế hệ đã tồn tại, thao tác upsert bị chặn hoàn toàn để ngăn chặn việc hồi sinh (resurrection) các thực thể đã xóa từ các sự kiện cũ replayed.
- **Định danh Thế hệ (Generation ID)**: `generation_id` được biểu diễn dưới dạng chuỗi nối canonical bằng ký tự hai chấm: `file_id:content_hash:parser_version:schema_version` (KHÔNG phải là một mã băm mật mã học). Cơ chế này cho phép các thế hệ sự kiện khác nhau (khi mã nguồn hoặc version của parser thay đổi) có thể hoạt động độc lập và tự do ghi đè, trong khi chặn các stale replay của chính thế hệ đó.
- **Giới hạn Cô lập Lỗi Mixed-Batch (Accepted Limitation)**: Do Neo4j Kafka Sink Connector mặc định thực thi toàn bộ bản ghi trong cùng một batch window thành một transaction duy nhất, bất kỳ lỗi runtime nào của một bản ghi (ví dụ: endpoint mismatch gây ra lỗi chia cho 0) cũng sẽ khiến toàn bộ batch transaction bị rollback ở phía Neo4j. Connector vẫn tiếp tục chạy (`RUNNING`) và đẩy bản ghi lỗi sang DLQ (`connector.errors`), nhưng các bản ghi hợp lệ trong cùng batch đó sẽ bị mất và cần phải được replay/retry lại để ghi thành công.